# 00_prepare — EV-10 · RT-08 데이터 준비

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 2절의 설계를 표로 만든다. CPU로 몇 분 걸린다.

- A 분할에 v2 채점 정정 반영 (데이터 생성 코드의 로더를 그대로 사용)
- 원천별 모델 집단 고정 (train 통계만)
- 질문마다 집단별 풀 확률, 3단계 라벨(τ = .8), 서비스 비용 1:3:5
- LLMRouterBench·SPROUT를 train 50 / dev 15 / test 35로 다시 나눔. RouterArena와 겹치는 문항은 test
- RT-08용 기존 정의 라벨(v1·v2)

test 행은 기록만 하고 이 노트북에서 요약하지 않는다.

In [ ]:
from pathlib import Path
import json, sys

# 로컬: 저장소 checkout과 git에 없는 데이터가 있는 checkout. Colab에서는 Drive 경로로 바꾼다.
REPO = Path.cwd().resolve().parents[2] if Path.cwd().name == 'router_final' else Path.cwd()
DATA_ROOT = REPO  # ml/artifacts, ml/data/raw 가 있는 곳
OUTPUT_DIR = REPO / 'ml/artifacts/router-final-v1/prepare'
print(REPO, DATA_ROOT, OUTPUT_DIR)

In [ ]:
sys.path.insert(0, str(REPO / 'ml/notebooks/router_final'))
import prepare
assert not (OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir())), '새 빈 폴더에만 쓴다'
summary = prepare.main(['--config', str(REPO / 'ml/notebooks/router_final/config.json'), '--repo', str(REPO),
                        '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR)])

In [ ]:
import pandas as pd
print(pd.DataFrame(summary['rows']).T)
print(pd.DataFrame(summary['needed_tier_mix_train']).T)
print('RouterArena 겹침(test로 보냄):', summary['arena_overlap_rows'])
for source, tiers in summary['groups'].items():
    print(source, tiers)

## 다음

`manifest.json`의 출력 SHA를 확인한 뒤 `OUTPUT_DIR`을 Drive `LLM_Router/router-final/runs/run001/prepare/`에 올린다. 01_embed부터는 이 폴더를 입력으로 쓴다.